# C — Model akhir & submission (Seno)

Notebook bagian C, Task 2 Datavidia 11.0.

**Alur:** label kejadian dari B (`fahmi/v1`) → seed label → label spreading → Logistic Regression dari **embedding per foto saja** (tanpa `event_id`) → prediksi 3.180 foto → submission.

**Output** (folder run baru, tidak menimpa): `model.joblib`, `pred_train.csv`, `submission_runN.csv`, `meta.json`.

Tidak ada label manual. Nama kelas hanya berasal dari `rules.json` lewat B.

## 0. Library & versi

In [1]:
import json, os, platform
from datetime import datetime

import numpy as np
import pandas as pd
import pyarrow, scipy, sklearn, joblib

from sklearn.decomposition import PCA
from sklearn.semi_supervised import LabelSpreading
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import accuracy_score, f1_score

VERSI_LIB = {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
             "pyarrow": pyarrow.__version__, "scipy": scipy.__version__,
             "scikit-learn": sklearn.__version__, "joblib": joblib.__version__}
for k, v in VERSI_LIB.items():
    print(f"{k:13s} {v}")

python        3.13.18
numpy         2.5.3
pandas        3.0.5
pyarrow       26.0.1
scipy         1.18.1
scikit-learn  1.9.1
joblib        1.6.0


## Pengaturan
Semua path dan angka di sini. Path relatif terhadap folder `seno/`.

In [2]:
# --- Path input ---
KANDIDAT_SAMPLE_SUB = ["../../Task-2/sample_submission.csv",   # susunan laptop Seno
                       "../../sample_submission.csv",          # susunan laptop Fahmi
                       "../Dataset/sample_submission.csv"]     # susunan README repo
FITUR_DIR = "../hafizh/v2"     # embedding DINOv2-L (harus sama dengan input B)
LABEL_DIR = "../fahmi/v1"      # events.parquet + event_names.csv dari B

# --- Output (jika sudah ada, dipakai run2, run3, ... ; tidak menimpa) ---
OUTPUT_DIR = "run1"

SEED = 42
KELAS = ["Earthquake", "Explosion", "Fire", "Hurricane", "Tsunami", "Volcano"]

# --- Ruang seed (harus sama persis dengan B, supaya is_core B tereproduksi) ---
B_PCA = 64
B_ROUND = 4
CORE_FRAC = 0.5

# --- Sumber seed label ---
#   "is_core"             : is_core dari B (cluster lemah tidak punya seed)
#   "pusat_semua_cluster" : CORE_FRAC foto terdekat pusat dari SEMUA cluster
#                           (aturan is_core B tanpa filter cluster lemah)
SUMBER_LABEL = "pusat_semua_cluster"

# --- Label spreading ---
LS_KNN = 15
LS_ALPHA = 0.2

# --- Classifier (fitur per foto saja, tanpa event_id) ---
FITUR_BLOK = ["color_pre", "color_post"]   # file dino_<blok>.npy
C_PCA = 128
LR_C = 1.0
CV_FOLD = 5

SAMPLE_SUB = next((p for p in KANDIDAT_SAMPLE_SUB if os.path.exists(p)), None)
np.random.seed(SEED)
for p in [SAMPLE_SUB, FITUR_DIR, LABEL_DIR]:
    print(f"{'ada ' if p and os.path.exists(p) else 'TIDAK ADA'}  {p}")
assert SAMPLE_SUB is not None, "sample_submission.csv tidak ketemu: tambahkan path-nya ke KANDIDAT_SAMPLE_SUB"
assert SUMBER_LABEL in ("is_core", "pusat_semua_cluster")

ada   ../../Task-2/sample_submission.csv
ada   ../hafizh/v2
ada   ../fahmi/v1


## 1. Muat data & cek input

In [3]:
sub    = pd.read_csv(SAMPLE_SUB)
events = pd.read_parquet(f"{LABEL_DIR}/events.parquet")
names  = pd.read_csv(f"{LABEL_DIR}/event_names.csv").set_index("event_id")
with open(f"{LABEL_DIR}/meta.json", encoding="utf-8") as f:
    meta_b = json.load(f)

ids = sub["Id"].to_numpy()
N = len(ids)
KOLOM_TARGET = sub.columns[1]

# Urutan Id & nama kelas
assert sub["Id"].is_unique and KOLOM_TARGET == "Phenomenon"
assert len(events) == N and (events["Id"].to_numpy() == ids).all(), "urutan Id events.parquet beda dengan sample_submission"
assert set(names["Phenomenon"]) <= set(KELAS), "ada nama kelas di luar 6 kelas resmi"
assert set(events["event_id"]) == set(names.index)
assert meta_b["rules"]["status"] == "dikunci", "rules.json B belum dikunci"

# Fitur yang dipakai B harus sama dengan FITUR_DIR
assert meta_b["input"]["fitur"] == FITUR_DIR.replace("../", ""), f"B memakai {meta_b['input']['fitur']}, bukan {FITUR_DIR}"

# Embedding
emb = {}
for b in sorted(set(FITUR_BLOK) | {"color_pre", "color_post"}):
    emb[b] = np.load(f"{FITUR_DIR}/dino_{b}.npy")
    assert emb[b].shape == (N, 1024) and not np.isnan(emb[b]).any(), f"embedding {b} bermasalah"

event_id = events["event_id"].to_numpy()
nama_cluster = names["Phenomenon"].reindex(event_id).to_numpy()   # nama kelas per foto dari cluster-nya

print("N =", N, "| cluster:", len(names), "| cluster lemah:", int(names["is_weak"].sum()))
print("Rules B:", meta_b["rules"])
print("Semua cek input: LULUS")

N = 3180 | cluster: 20 | cluster lemah: 16
Rules B: {'status': 'dikunci', 'dikunci_pada': '2026-11-27 18:28'}
Semua cek input: LULUS


In [4]:
# Ringkasan output B per kelas
ringkas_b = pd.DataFrame({
    "n_cluster": names["Phenomenon"].value_counts(),
    "n_cluster_lemah": names.loc[names["is_weak"], "Phenomenon"].value_counts(),
    "n_foto": pd.Series(nama_cluster).value_counts(),
    "n_is_core_B": pd.Series(nama_cluster[events["is_core"].to_numpy()]).value_counts(),
}).reindex(KELAS).fillna(0).astype(int)
ringkas_b

,n_cluster,n_cluster_lemah,n_foto,n_is_core_B
Earthquake,2,2,158,0
Explosion,6,5,887,20
Fire,3,3,369,0
Hurricane,1,1,112,0
Tsunami,7,4,1481,327
Volcano,1,1,173,0


## 2. Seed label
Ruang `Z` dibangun persis seperti B: L2 per blok → gabung pre+post → kurangi rata-rata → PCA 64 → bulatkan 4.
Cek: `is_core` B harus keluar sama 100% dari ruang ini.

In [5]:
def l2(a):
    a = a.astype(np.float64)
    return a / np.linalg.norm(a, axis=1, keepdims=True)

X_b = np.hstack([l2(emb["color_pre"]), l2(emb["color_post"])])
X_b = X_b - X_b.mean(axis=0)
Z = np.round(PCA(B_PCA, random_state=SEED).fit_transform(X_b), B_ROUND)

def pilih_seed(lewati_lemah):
    # CORE_FRAC foto terdekat pusat cluster (ruang Z), sama dengan langkah is_core di B
    seed = np.zeros(N, dtype=bool)
    for e in names.index:
        if lewati_lemah and names.loc[e, "is_weak"]:
            continue
        idx = np.where(event_id == e)[0]
        jarak = np.linalg.norm(Z[idx] - Z[idx].mean(axis=0), axis=1)
        seed[idx[np.argsort(jarak, kind="stable")[:int(round(CORE_FRAC * len(idx)))]]] = True
    return seed

core_ulang = pilih_seed(lewati_lemah=True)
assert (core_ulang == events["is_core"].to_numpy()).all(), "ruang Z tidak sama dengan B (is_core beda)"
print("is_core B tereproduksi 100%")

SEED_MASK = {"is_core": core_ulang, "pusat_semua_cluster": pilih_seed(lewati_lemah=False)}
is_seed = SEED_MASK[SUMBER_LABEL]
print(f"SUMBER_LABEL = {SUMBER_LABEL} -> {int(is_seed.sum())} foto seed")

is_core B tereproduksi 100%
SUMBER_LABEL = pusat_semua_cluster -> 1588 foto seed


## 3. Label spreading
Graf kNN di ruang `Z`. Foto seed diberi nama kelas cluster-nya; foto lain (-1) diisi oleh label spreading.

In [6]:
def sebar(is_seed):
    y = np.full(N, -1)
    y[is_seed] = pd.Categorical(nama_cluster[is_seed], categories=KELAS).codes
    ls = LabelSpreading(kernel="knn", n_neighbors=LS_KNN, alpha=LS_ALPHA, max_iter=1000).fit(Z, y)
    kelas_ls = np.array(KELAS)[ls.classes_]          # hanya kelas yang punya seed
    dist = ls.label_distributions_
    return kelas_ls[dist.argmax(axis=1)], dist.max(axis=1)

# Perbandingan dua sumber seed (diagnostik, untuk diskusi tim)
banding = {}
for mode, mask in SEED_MASK.items():
    lab, _ = sebar(mask)
    banding[f"seed_{mode}"] = pd.Series(nama_cluster[mask]).value_counts()
    banding[f"hasil_{mode}"] = pd.Series(lab).value_counts()
display(pd.DataFrame(banding).reindex(KELAS).fillna(0).astype(int))

y_latih, yakin = sebar(is_seed)
print(f"Dipakai: {SUMBER_LABEL} | kelas terisi: {len(set(y_latih))}/6 | "
      f"sama dengan nama cluster: {(y_latih == nama_cluster).mean():.3f}")
if len(set(y_latih)) < len(KELAS):
    print("PERINGATAN: ada kelas tanpa label latih -> classifier tidak akan pernah memprediksinya")

,seed_is_core,hasil_is_core,seed_pusat_semua_cluster,hasil_pusat_semua_cluster
Earthquake,0,0,79,144
Explosion,20,431,443,877
Fire,0,0,185,370
Hurricane,0,0,56,98
Tsunami,327,2749,739,1507
Volcano,0,0,86,184


Dipakai: pusat_semua_cluster | kelas terisi: 6/6 | sama dengan nama cluster: 0.940


## 4. Classifier
Fitur per foto = blok embedding `FITUR_BLOK` (L2 per blok, digabung). **Tidak** memakai `event_id`.
Pipeline: StandardScaler → PCA → LogisticRegression (`class_weight="balanced"`).

CV di bawah mengukur seberapa konsisten classifier terhadap label latih, **bukan** skor leaderboard (tidak ada label asli).

In [7]:
def bangun_fitur(emb, blok):
    return np.hstack([l2(emb[b]) for b in blok])

def buat_pipeline():
    return Pipeline([
        ("scaler", StandardScaler()),
        ("pca", PCA(C_PCA, random_state=SEED)),
        ("clf", LogisticRegression(class_weight="balanced", C=LR_C, max_iter=5000)),
    ])

X_c = bangun_fitur(emb, FITUR_BLOK)
print("Fitur classifier:", X_c.shape)

cv = StratifiedKFold(CV_FOLD, shuffle=True, random_state=SEED)
pred_cv = cross_val_predict(buat_pipeline(), X_c, y_latih, cv=cv)
CV = {"acc": round(float(accuracy_score(y_latih, pred_cv)), 4),
      "macro_f1": round(float(f1_score(y_latih, pred_cv, average="macro")), 4)}
print(f"CV {CV_FOLD}-fold vs label latih: acc {CV['acc']:.3f} | macro-F1 {CV['macro_f1']:.3f}")

model = buat_pipeline().fit(X_c, y_latih)
pred = model.predict(X_c)
proba = model.predict_proba(X_c)
print("Urutan kelas model:", [str(k) for k in model.classes_])

Fitur classifier: (3180, 2048)


CV 5-fold vs label latih: acc 0.841 | macro-F1 0.798


Urutan kelas model: ['Earthquake', 'Explosion', 'Fire', 'Hurricane', 'Tsunami', 'Volcano']


In [8]:
HASIL = {
    "kelas_terprediksi": int(len(set(pred))),
    "sama_label_latih": round(float((pred == y_latih).mean()), 4),
    "sama_nama_cluster": round(float((pred == nama_cluster).mean()), 4),
}
print(HASIL)
distribusi = pd.DataFrame({
    "nama_cluster_B": pd.Series(nama_cluster).value_counts(),
    "label_latih": pd.Series(y_latih).value_counts(),
    "prediksi": pd.Series(pred).value_counts(),
}).reindex(KELAS).fillna(0).astype(int)
distribusi["porsi_prediksi"] = (distribusi["prediksi"] / N).round(3)
distribusi

{'kelas_terprediksi': 6, 'sama_label_latih': 0.933, 'sama_nama_cluster': 0.8975}


,nama_cluster_B,label_latih,prediksi,porsi_prediksi
Earthquake,158,144,144,0.045
Explosion,887,877,905,0.285
Fire,369,370,427,0.134
Hurricane,112,98,98,0.031
Tsunami,1481,1507,1419,0.446
Volcano,173,184,187,0.059


In [9]:
# Baris = nama cluster dari B, kolom = prediksi classifier
pd.crosstab(pd.Series(nama_cluster, name="nama_cluster_B"), pd.Series(pred, name="prediksi"))

prediksi,Earthquake,Explosion,Fire,Hurricane,Tsunami,Volcano
nama_cluster_B,,,,,,
Earthquake,139,3,11,0,2,3
Explosion,1,814,5,1,63,3
Fire,3,1,334,0,27,4
Hurricane,0,2,3,91,8,8
Tsunami,1,85,64,6,1316,9
Volcano,0,0,10,0,3,160


## 5. Simpan output
Folder run baru; jika `OUTPUT_DIR` sudah ada, dipakai nomor berikutnya (tidak menimpa).

In [10]:
nomor = int(OUTPUT_DIR.lstrip("run"))
OUT = f"run{nomor}"
while os.path.exists(OUT):
    nomor += 1
    OUT = f"run{nomor}"
os.makedirs(OUT)

# Model: pipeline (scaler, PCA, classifier) + urutan blok fitur + urutan kelas
artefak_model = {
    "pipeline": model,
    "fitur_blok": FITUR_BLOK,
    "l2_per_blok": True,
    "kelas": [str(k) for k in model.classes_],
    "versi_lib": VERSI_LIB,
}
joblib.dump(artefak_model, f"{OUT}/model.joblib")

# Prediksi saat training (acuan cek 100% di 02_submission)
pred_train = pd.DataFrame({"Id": ids, "Phenomenon": pred, "label_latih": y_latih,
                           "is_seed": is_seed, "event_id": event_id})
for i, k in enumerate(model.classes_):
    pred_train[f"proba_{k}"] = proba[:, i].round(6)
pred_train.to_csv(f"{OUT}/pred_train.csv", index=False)

# Submission: urutan & kolom persis sample_submission
submission = sub[["Id"]].copy()
submission[KOLOM_TARGET] = pred
submission.to_csv(f"{OUT}/submission_{OUT}.csv", index=False)

meta = {
    "versi": OUT,
    "oleh": "Seno",
    "tanggal": datetime.now().strftime("%Y-%m-%d %H:%M"),
    "input": {"fitur": FITUR_DIR.replace("../", ""), "label": LABEL_DIR.replace("../", ""),
              "rules": meta_b["rules"]},
    "pengaturan": {"sumber_label": SUMBER_LABEL, "core_frac": CORE_FRAC, "b_pca": B_PCA, "b_round": B_ROUND,
                   "ls_kernel": "knn", "ls_knn": LS_KNN, "ls_alpha": LS_ALPHA,
                   "fitur_blok": FITUR_BLOK, "c_pca": C_PCA, "lr_C": LR_C,
                   "class_weight": "balanced", "seed": SEED},
    "n_seed": int(is_seed.sum()),
    "cv_vs_label_latih": CV,
    "hasil": HASIL,
    "distribusi_prediksi": {k: int(distribusi.loc[k, "prediksi"]) for k in KELAS},
    "versi_lib": VERSI_LIB,
    "catatan": f"{SUMBER_LABEL}, LogReg balanced, fitur {'+'.join(FITUR_BLOK)}",
}
with open(f"{OUT}/meta.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)
print("Disimpan ke", OUT, ":", sorted(os.listdir(OUT)))

Disimpan ke run1 : ['meta.json', 'model.joblib', 'pred_train.csv', 'submission_run1.csv']


In [11]:
# Cek ulang dari file: muat model -> prediksi ulang tanpa .fit() -> harus sama 100%
m = joblib.load(f"{OUT}/model.joblib")
pred_ulang = m["pipeline"].predict(bangun_fitur(emb, m["fitur_blok"]))
cek_sub = pd.read_csv(f"{OUT}/submission_{OUT}.csv")
assert (pred_ulang == pred).all(), "prediksi dari model tersimpan beda"
assert list(cek_sub.columns) == list(sub.columns) and (cek_sub["Id"] == sub["Id"]).all()
assert cek_sub[KOLOM_TARGET].notna().all() and set(cek_sub[KOLOM_TARGET]) <= set(KELAS)
print("Prediksi model tersimpan = prediksi training: 100%")
print("Format submission: LULUS")
print(f"Pesan commit: [C] {OUT}: LogReg, {SUMBER_LABEL}")

Prediksi model tersimpan = prediksi training: 100%
Format submission: LULUS
Pesan commit: [C] run1: LogReg, pusat_semua_cluster
